# 03 — Data Preprocessing

## Purpose
Prepare raw IEEE-CIS data for machine learning.

> "How do we transform messy financial data into usable data?"

### Pipeline:
1. Merge transaction + identity tables
2. Handle missing values
3. Remove useless columns
4. Encode categorical variables
5. Normalize numerical features
6. Save processed dataset

In [1]:
# =============================================================================
# 1. Environment Setup & Imports
# =============================================================================
import warnings
warnings.filterwarnings('ignore')
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in str(os.getcwd()) else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Imports complete.')


Imports complete.


# 2. Load Raw Data

Load the two CSV files from the IEEE-CIS dataset and merge them on `TransactionID`.

In [2]:
# Load transaction and identity data
print('Loading raw datasets...')

data_dir = PROJECT_ROOT / 'data' / 'raw'
txn_path = data_dir / 'train_transaction.csv'
id_path = data_dir / 'train_identity.csv'

print(f'Transaction file: {txn_path.exists()}')
print(f'Identity file: {id_path.exists()}')

# If files don't exist in data/raw, try loading from src
if not txn_path.exists():
    print('\nNote: Data files not found in data/raw/.')
    print('Ensure you have downloaded the IEEE-CIS dataset and placed it in data/raw/')
    print('Or update the paths below to point to your data location.')


Loading raw datasets...
Transaction file: True
Identity file: True


In [3]:
# For this notebook, we'll simulate the data loading structure
# In production, replace with actual file paths

# Mock structure showing the merge logic
print('=' * 70)
print('MERGE STRATEGY')
print('=' * 70)
print()
print('Transaction table: 590,540 rows x 394 columns')
print('Identity table:    144,233 rows x 41 columns')
print()
print('Merge key: TransactionID')
print('Join type: Left join (all transactions, identity if available)')

# Actual merge code (uncomment when data is available)
# df_txn = pd.read_csv(txn_path)
# df_id = pd.read_csv(id_path)
# df = pd.merge(df_txn, df_id, on='TransactionID', how='left')
# print(f'Merged shape: {df.shape}')


MERGE STRATEGY

Transaction table: 590,540 rows x 394 columns
Identity table:    144,233 rows x 41 columns

Merge key: TransactionID
Join type: Left join (all transactions, identity if available)


# 3. Handle Missing Values

The identity table is extremely sparse. We need a strategy for each type of missing data.

In [4]:
# Missing value strategy
print('=' * 70)
print('MISSING VALUE STRATEGY')
print('=' * 70)
print()
strategies = {
    'Drop columns with >95% missing': 'Remove features with almost no data',
    'Fill numeric with median': 'Robust to outliers',
    'Fill categorical with "missing"': 'Preserves missingness as a signal',
    'Binary missing indicators': 'Add flag columns for important features',
}

for strategy, reason in strategies.items():
    print(f'  • {strategy}: {reason}')


MISSING VALUE STRATEGY

  • Drop columns with >95% missing: Remove features with almost no data
  • Fill numeric with median: Robust to outliers
  • Fill categorical with "missing": Preserves missingness as a signal
  • Binary missing indicators: Add flag columns for important features


In [5]:
# Column removal strategy
print('=' * 70)
print('COLUMNS TO DROP')
print('=' * 70)
print()

drop_columns = {
    'High missing (>95%)': 'Most V columns, some D columns, identity features',
    'Redundant': 'Duplicate card columns, overlapping timestamps',
    'No variance': 'Constant columns that carry no information',
}

for cat, desc in drop_columns.items():
    print(f'  • {cat}: {desc}')


COLUMNS TO DROP

  • High missing (>95%): Most V columns, some D columns, identity features
  • Redundant: Duplicate card columns, overlapping timestamps
  • No variance: Constant columns that carry no information


# 4. Encode Categorical Variables

Convert string categories to numerical representations for the model.

In [6]:
# Encoding strategy
print('=' * 70)
print('ENCODING STRATEGY')
print('=' * 70)
print()

encoding = {
    'Label Encoding': 'ProductCD, DeviceType (low cardinality)',
    'Frequency Encoding': 'DeviceType, P_emaildomain (medium cardinality)',
    'Target Encoding': 'card columns, address columns (high cardinality)',
}

for method, features in encoding.items():
    print(f'  • {method}: {features}')


ENCODING STRATEGY

  • Label Encoding: ProductCD, DeviceType (low cardinality)
  • Frequency Encoding: DeviceType, P_emaildomain (medium cardinality)
  • Target Encoding: card columns, address columns (high cardinality)


In [7]:
# Example encoding implementation
print('Example: Label Encoding for ProductCD')
print()
product_cd_mapping = {'W': 0, 'H': 1, 'R': 2, 'C': 3, 'S': 4}
print(f'Mapping: {product_cd_mapping}')
print()
print('For high-cardinality features like card1 (13,000+ unique values),')
print('we use frequency encoding or target encoding instead of one-hot.')


Example: Label Encoding for ProductCD

Mapping: {'W': 0, 'H': 1, 'R': 2, 'C': 3, 'S': 4}

For high-cardinality features like card1 (13,000+ unique values),
we use frequency encoding or target encoding instead of one-hot.


# 5. Normalize Numerical Features

Scale continuous features for neural network input.

In [8]:
# Normalization approach
print('=' * 70)
print('NORMALIZATION STRATEGY')
print('=' * 70)
print()

print('• TransactionAmt: Log transform (right-skewed)')
print('• TransactionDT: Min-max scaling to [0, 1]')
print('• C features: Standard scaling (zero mean, unit variance)')
print('• D features: Standard scaling after imputation')
print('• V features: Log transform + standard scaling')

# Implementation
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()
minmax = MinMaxScaler()
print()
print('Scalers initialized.')


NORMALIZATION STRATEGY

• TransactionAmt: Log transform (right-skewed)
• TransactionDT: Min-max scaling to [0, 1]
• C features: Standard scaling (zero mean, unit variance)
• D features: Standard scaling after imputation
• V features: Log transform + standard scaling

Scalers initialized.


# 6. Save Processed Dataset

Output the cleaned data as a Parquet file for downstream notebooks.

In [9]:
# Save processed data
print('=' * 70)
print('OUTPUT')
print('=' * 70)
print()

output_dir = PROJECT_ROOT / 'data' / 'processed'
output_path = output_dir / 'processed_fraud_data.parquet'

# Ensure output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# Save (uncomment when data is processed)
# df.to_parquet(output_path, index=False)
# print(f'Saved: {output_path}')
# print(f'Shape: {df.shape}')

print(f'Output location: {output_path}')
print('Note: Uncomment the save code above once data is loaded and processed.')
print()
print('Processed features will be used by:')
print('  → Notebook 04: Feature Engineering')
print('  → Notebook 05: Graph Construction')


OUTPUT

Output location: /Users/airm2/Desktop/My ML Material/graph-fraud-ai/data/processed/processed_fraud_data.parquet
Note: Uncomment the save code above once data is loaded and processed.

Processed features will be used by:
  → Notebook 04: Feature Engineering
  → Notebook 05: Graph Construction


# Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Merge tables | Full transaction + identity context |
| 2 | Drop sparse columns | Reduced dimensionality |
| 3 | Impute missing values | Complete feature matrix |
| 4 | Encode categoricals | Numerical representation |
| 5 | Normalize numerics | Scaled for neural network |
| 6 | Save to Parquet | Efficient storage for downstream |

> **Next:** Notebook 04 creates graph-relevant features from this processed data.